Pablo Méndez, Rodrigo Dueñas, Roberto Howe and Jordi Casaldàliga

### 1. Importing Libraries

We’ll start by importing the necessary Python libraries.  
We mainly use **pandas** for data manipulation and **os** for handling file paths.

In [4]:
import pandas as pd
import os
from pathlib import Path
from datetime import datetime
from tabulate import tabulate

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import argparse

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

### 2. Defining the Data Paths

Here we specify the paths to the two folders that contain our CSV files.  
The idea is to automatically go through each file in the main archive and check whether a matching file exists in the secondary one.

In [6]:
secondary_archive_path  = '../data/mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16'
main_archive_path = '../data/mturkfitbit_export_4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16'
full_path = '../data/full_path'
if not os.path.exists(full_path):
    os.makedirs(full_path)

### 3. Listing Files

We now list all the `.csv` files in the main folder.  
This will help us iterate automatically over them and perform the merge operation for each.

In [7]:
main_files = [f for f in os.listdir(main_archive_path) if f.endswith('.csv')]
main_files

['dailyActivity_merged.csv',
 'dailyCalories_merged.csv',
 'dailyIntensities_merged.csv',
 'dailySteps_merged.csv',
 'heartrate_seconds_merged.csv',
 'hourlyCalories_merged.csv',
 'hourlyIntensities_merged.csv',
 'hourlySteps_merged.csv',
 'minuteCaloriesNarrow_merged.csv',
 'minuteCaloriesWide_merged.csv',
 'minuteIntensitiesNarrow_merged.csv',
 'minuteIntensitiesWide_merged.csv',
 'minuteMETsNarrow_merged.csv',
 'minuteSleep_merged.csv',
 'minuteStepsNarrow_merged.csv',
 'minuteStepsWide_merged.csv',
 'sleepDay_merged.csv',
 'weightLogInfo_merged.csv']

### 4. Merging the Files

For each CSV file in the main archive:
1. We read it into a DataFrame.
2. We check if there’s a corresponding file with the same name in the secondary archive.
3. If it exists, we concatenate both DataFrames.
4. We remove duplicates (to avoid repeated records).
5. We save the merged DataFrame as a new CSV, with another name.

In [ ]:
for file in main_files:
    # Read the CSV file from the main archive
    main_df = pd.read_csv(os.path.join(main_archive_path, file))
    secondary_file_path = os.path.join(secondary_archive_path, file)

    if os.path.exists(secondary_file_path):
        # Read the corresponding CSV file from the secondary archive
        secondary_df = pd.read_csv(secondary_file_path)

        # Merge the two dataframes
        merged_df = pd.concat([main_df, secondary_df], ignore_index=True)

        #remove duplicates if needed
        merged_df = merged_df.drop_duplicates()

        # Save the merged dataframe
        merged_df.to_csv(os.path.join(full_path, f'full_{file}'), index=False)
        print(f"Merged {file} with its secondary counterpart.")
    else:
        main_df.to_csv(os.path.join(full_path, f'full_{file}'), index=False)
        print(f"No secondary file found for {file}. Skipping merge.")

No secondary file found for minuteIntensitiesWide_merged.csv. Skipping merge.
Merged dailyActivity_merged.csv with its secondary counterpart.
No secondary file found for sleepDay_merged.csv. Skipping merge.
Merged heartrate_seconds_merged.csv with its secondary counterpart.
Merged minuteIntensitiesNarrow_merged.csv with its secondary counterpart.
Merged minuteMETsNarrow_merged.csv with its secondary counterpart.
Merged hourlySteps_merged.csv with its secondary counterpart.
Merged hourlyIntensities_merged.csv with its secondary counterpart.
No secondary file found for dailyCalories_merged.csv. Skipping merge.
Merged hourlyCalories_merged.csv with its secondary counterpart.
Merged minuteSleep_merged.csv with its secondary counterpart.
Merged minuteCaloriesNarrow_merged.csv with its secondary counterpart.
No secondary file found for dailySteps_merged.csv. Skipping merge.
No secondary file found for minuteCaloriesWide_merged.csv. Skipping merge.
Merged minuteStepsNarrow_merged.csv with its

### 5. Helper Functions

To make our code modular and readable, we’ll define several helper functions that handle small, specific tasks such as formatting dates, numbers, and summarizing columns.

---

##### `format_date(date_str)`

This function tries to interpret strings as dates.  
Some Fitbit files store timestamps in different formats (with or without hours), so we check both.

---

##### `format_number(value)`

This function ensures numeric outputs (like averages or standard deviations)  
are printed with **three decimal places**.

---

##### `process_col(column, serie)`

This function calculates **descriptive statistics** for one column:
- Count of non-null values  
- Number of missing values  
- Number of distinct values  
- (If numeric) → min, max, mean, and standard deviation  

It returns a dictionary containing these statistics, ready to be turned into a table.

---

##### `log_row(text, fp)`

A simple utility that writes text to both:
- the **console** (so we see progress live),
- and a **log file** (to keep results saved).


In [ ]:
def format_date(date_str):
    ret = pd.NaT
    try:
        ret = datetime.strptime(date_str, '%m/%d/%Y %I:%M:%S %p')
    except ValueError:
        pass

    try:
        ret = datetime.strptime(date_str, '%m/%d/%Y')
    except ValueError:
        pass

    return ret

def format_number(value):
    if isinstance(value, (int, float)):
        return f"{value:.3f}"
    return value

def process_col(column, serie):
    row = {'Column': column}

    row.update({
        'Count': format_number(serie.count()),
        'Null Count': format_number(serie.isnull().sum()),
        'Distinct Count': format_number(serie.nunique())
    })

    if pd.api.types.is_numeric_dtype(serie):
      row.update({
          'Min': format_number(serie.min()),
          'Max': format_number(serie.max()),
          'Average': format_number(serie.mean()),
          'Std Dev': format_number(serie.std())
      })

    return row

def log_row(text, fp):
  print(text)
  fp.write(text + "\n")

## 6. Clean the data

Now that we have helper functions, we can define the main data-cleaning function.

### What it does:
1. Opens a report file (`table_statistics.txt`).
2. Loops through every `.csv` file in the dataset folder.
3. Loads the file into a DataFrame.
4. Detects and converts columns that look like dates.
5. Computes per-column statistics using `process_col()`.
6. Logs those statistics as a formatted table using `tabulate`.
7. Saves a cleaned version of the CSV into a new folder.

In [ ]:
def main(datadir, cleandir):
  with open('table_statistics.txt', 'w') as f:
    for filename in os.listdir(datadir):
      if not filename.endswith('.csv'): continue

      file_path = os.path.join(datadir, filename)
      df = pd.read_csv(file_path)

      log_row(f"Statistics for {filename}:", f)

      stats = []
      date_columns_modified = False

      for column in df.columns:
        if df[column].dtype == 'object' and df[column].apply(lambda x: isinstance(x, str)).all():
          try:
            date_column = df[column].apply(format_date)
            if not date_column.isnull().all():
              df[column] = date_column
          except ValueError:
            pass

        row = process_col(column, df[column])
        stats.append(row)

      log_row(tabulate(stats, headers="keys", tablefmt="grid"), f)

      new_file_path = os.path.join(cleandir, f"{filename}")
      df.to_csv(new_file_path, index=False)

      f.flush()

if __name__ == "__main__":
    data ="/content/drive/MyDrive/Colab Notebooks/codeDL/full_path"
    clean = '/content/drive/MyDrive/Colab Notebooks/codeDL/full_path_clean'

    if not os.path.exists(clean):
      os.makedirs(clean)

    main(data, clean)

Statistics for full_heartrate_seconds_merged.csv:
+----------+-------------+--------------+------------------+--------------+---------------+--------------+--------------+
| Column   |       Count |   Null Count |   Distinct Count |          Min |           Max |      Average |      Std Dev |
+==========+=============+==============+==================+==============+===============+==============+==============+
| Id       | 3.61492e+06 |            0 |     15           |  2.02248e+09 |   8.87769e+09 |  5.46144e+09 |  1.97837e+09 |
+----------+-------------+--------------+------------------+--------------+---------------+--------------+--------------+
| Time     | 3.61492e+06 |            0 |      1.45622e+06 |              |               |              |              |
+----------+-------------+--------------+------------------+--------------+---------------+--------------+--------------+
| Value    | 3.61492e+06 |            0 |    168           | 36           | 203           | 78.1

#T1

Completed functions** --> Revisarlo

In [ ]:
def prepare_sleep_prediction_data(activity_df, sleep_df):
    """
    Prepare features and target for sleep prediction
    activity of day D -> minutes asleep in the night after D (the sleep that ends on D + 1)
    """
    activity_df = activity_df.copy()
    sleep_df = sleep_df.copy()

    # Dates as real dates (in the cleaned CSVs they are text such as '2016-04-12')
    activity_df['ActivityDate'] = pd.to_datetime(activity_df['ActivityDate'])
    sleep_df['SleepDay'] = pd.to_datetime(sleep_df['SleepDay'])

    # SleepDay is the date on which the sleep ENDS, so the night after day D has SleepDay = D + 1
    activity_df['NextSleepDay'] = activity_df['ActivityDate'] + pd.to_timedelta(1, unit='D')

    # Merge activity and sleep data
    df = activity_df.merge(sleep_df, left_on=['Id', 'NextSleepDay'], right_on=['Id', 'SleepDay'], how='inner')

    # Select features to use
    features = [
        'TotalSteps', 'TotalDistance',
        'VeryActiveDistance', 'ModeratelyActiveDistance', 'LightActiveDistance',
        'VeryActiveMinutes', 'FairlyActiveMinutes', 'LightlyActiveMinutes'
    ]

    # Select target to predict
    target = ['TotalMinutesAsleep']

    # Split features and target
    X = df[features].values
    y = df[target].values

    return X, y

Full code for the exercises --> pendiente de terminar a partir de predict_sleep

In [ ]:
# Duplicated Function from the cell above
def prepare_sleep_prediction_data(activity_df, sleep_df):
    """
    Prepare features and target for sleep prediction
    activity of day D -> minutes asleep in the night after D (the sleep that ends on D + 1)
    """
    activity_df = activity_df.copy()
    sleep_df = sleep_df.copy()

    # Dates as real dates (in the cleaned CSVs they are text such as '2016-04-12')
    activity_df['ActivityDate'] = pd.to_datetime(activity_df['ActivityDate'])
    sleep_df['SleepDay'] = pd.to_datetime(sleep_df['SleepDay'])

    # SleepDay is the date on which the sleep ENDS, so the night after day D has SleepDay = D + 1
    activity_df['NextSleepDay'] = activity_df['ActivityDate'] + pd.to_timedelta(1, unit='D')

    # Merge activity and sleep data
    df = activity_df.merge(sleep_df, left_on=['Id', 'NextSleepDay'], right_on=['Id', 'SleepDay'], how='inner')

    # Select features to use
    features = [
        'TotalSteps', 'TotalDistance',
        'VeryActiveDistance', 'ModeratelyActiveDistance', 'LightActiveDistance',
        'VeryActiveMinutes', 'FairlyActiveMinutes', 'LightlyActiveMinutes'
    ]

    # Select target to predict
    target = ['TotalMinutesAsleep']

    # Split features and target
    X = df[features].values
    y = df[target].values

    return X, y

def predict_sleep(model, scaler, features):
    """
    Make predictions using the trained model
    """
    # Scale features
    scaled_features = scaler.transform(features)

    # Convert to tensor and predict
    # TODO

    return prediction

class FitbitSleepPredictor(nn.Module):
    def __init__(self, input_size):
        """
        Initialize the FitbitSleepPredictor neural network.

        Args:
            input_size (int): The number of input features.
        """
        super(FitbitSleepPredictor, self).__init__()

        # Define the neural network architecture using a Sequential container
        #TODO

    def forward(self, x):
        """
        Defines the forward pass of the neural network.

        Args:
            x (torch.Tensor): Input tensor containing the features.

        Returns:
            torch.Tensor: The predicted sleep duration.
        """
        # Pass the input through the entire network and return the result
        return self.network(x)


def train_sleep_predictor(X, y, epochs=300, batch_size=32, learning_rate=0.001):
    """
    Train a neural network model to predict sleep duration based on Fitbit activity data.

    Args:
        X (numpy.ndarray): Input features (activity data)
        y (numpy.ndarray): Target values (sleep duration)
        epochs (int): Number of training epochs
        batch_size (int): Size of mini-batches for training

    Returns:
        tuple: Trained model and feature scaler
    """
    # Split the data into training and testing sets
    # 80% for training, 20% for testing
    # TODO

    # Initialize and fit the StandardScaler to normalize the feature data
    # TODO

    # Fit the scaler to the training data and transform it
    # TODO

    # Transform the test data using the fitted scaler
    # TODO

    # Convert numpy arrays to PyTorch tensors for use with PyTorch models
    # TODO


    # Create an instance of the FitbitSleepPredictor model
    # The input size is set to the number of features in the training data
    model = FitbitSleepPredictor(input_size=X_train.shape[1])

    # Define the loss function (Mean Squared Error for regression tasks)
    # TODO

    # Define the optimizer (Adam) with a learning rate of 0.001
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Initialize lists to store training and validation losses
    train_losses = []
    val_losses = []

    # Training loop
    for epoch in range(epochs):
        # Set the model to training mode (enables dropout, etc.)
        model.train()
        epoch_train_loss = 0

        # Mini-batch training

        for i in range(0, len(X_train), batch_size):
            # Extract a mini-batch of data
            batch_X = X_train[i:i+batch_size]
            batch_y = y_train[i:i+batch_size]

            # Forward pass: compute the model output for this batch
            outputs = model(batch_X)

            # Compute the loss between the predicted and actual values
            loss = criterion(outputs, batch_y)

            # Backward pass and optimization
            # Clear previous gradients
            # This is necessary because PyTorch accumulates gradients by default.
            # If we don't clear them, the gradients from the previous batch would be
            # added to the current batch's gradients, leading to incorrect updates.
            # By zeroing the gradients, we ensure that each batch starts with a clean slate.
            optimizer.zero_grad()

            loss.backward()        # Compute gradients
            optimizer.step()       # Update model parameters

            epoch_train_loss += loss.item()

        # Calculate average training loss for the epoch
        avg_train_loss = epoch_train_loss / (len(X_train) / batch_size)
        train_losses.append(avg_train_loss)

        # Validation
        model.eval()
        with torch.no_grad():
            val_predictions = model(X_test)
            val_loss = criterion(val_predictions, y_test)
            val_losses.append(val_loss.item())

        if epoch % 10 == 0:
            # Print the current epoch and test loss
            print(f'Epoch [{epoch}/{epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss.item():.4f}')

    # Plot training and validation loss
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.savefig('loss_plot.png')
    plt.close()

    # Return the trained model and the scaler for later use in predictions
    return model, scaler

def evaluate_model(model, X_test, y_test):
    model.eval()

    raise RuntimeError("Not implemented")
    # TODO


if __name__ == "__main__":

  # Set up argument parser
  parser = argparse.ArgumentParser(description='Train and evaluate a sleep prediction model')
  parser.add_argument('--datadir', type=str, default='../archive/clean', help='Directory containing the data files')
  parser.add_argument('--epochs', type=int, default=300, help='Number of training epochs')
  parser.add_argument('--batch_size', type=int, default=32, help='Size of mini-batches for training')
  parser.add_argument('--learning_rate', type=float, default=0.001, help='Learning rate for the optimizer')
  parser.add_argument('--test_size', type=float, default=0.2, help='Proportion of data to use for testing')
  parser.add_argument('--random_state', type=int, default=42, help='Random state for reproducibility')
  args = parser.parse_args()

# Load your data
  activity_df = pd.read_csv(os.path.join(args.datadir, 'dailyActivity_merged.csv'))
  sleep_df = pd.read_csv(os.path.join(args.datadir, 'sleepDay_merged.csv'))

  # Prepare data
  X, y = prepare_sleep_prediction_data(activity_df, sleep_df)

  # Train model
  model, scaler = train_sleep_predictor(X, y, epochs=args.epochs, batch_size=args.batch_size)

  # Evaluate model
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=args.test_size, random_state=args.random_state)
  X_test = torch.FloatTensor(scaler.transform(X_test))
  y_test = torch.FloatTensor(y_test)
  evaluate_model(model, X_test, y_test)

  # Make predictions
  new_features = np.array([[10000, 8.0, 2.0, 3.0, 3.0, 30, 45, 180]])  # Example input
  predicted_sleep = predict_sleep(model, scaler, new_features)
  print(f"Predicted sleep duration: {predicted_sleep[0][0]:.2f} minutes")



usage: colab_kernel_launcher.py [-h] [--datadir DATADIR] [--epochs EPOCHS]
                                [--batch_size BATCH_SIZE]
                                [--learning_rate LEARNING_RATE]
                                [--test_size TEST_SIZE]
                                [--random_state RANDOM_STATE]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-ca3fbff2-febd-4cb4-a838-f726a2410c55.json


SystemExit: 2

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
